# Exercise 3: LoRA Tuning

In this exercise, we will perform parameter-efficient fine-tuning using LoRA (Low-Rank Adaptation) or QLoRA (Quantized LoRA) on our base model. We'll use Hugging Face Transformers, PEFT, and bitsandbytes libraries for efficient training.

## Learning Objectives

By the end of this exercise, you will be able to:
- Understand the concept of parameter-efficient fine-tuning
- Configure LoRA/QLoRA for efficient model adaptation
- Implement a training loop with MLflow tracking
- Monitor GPU memory usage during training
- Save and load LoRA adapters

## Prerequisites

Before starting this exercise, ensure you have:
1. Completed Exercise 2: Data Preparation
2. A formatted and tokenized dataset ready for training
3. Access to a GPU (recommended for reasonable training times)
4. MLflow tracking server configured
5. Hugging Face token (if accessing private models or to avoid rate limits)

## Step 1: Import Required Libraries

Let's start by importing the necessary libraries for LoRA tuning:

## Step 0: Hugging Face Token Configuration

If you're accessing private models or want to avoid rate limits, you need to provide your Hugging Face token. You can get one from https://huggingface.co/settings/tokens

In [ ]:
# Get Hugging Face token
from getpass import getpass

# Option 1: Ask for token interactively (uncomment if running locally)
# hf_token = getpass("Enter your Hugging Face token: ")

# Option 2: Set token directly (for demonstration - replace with your token)
# hf_token = "your_hf_token_here"

# Option 3: Read from environment variable (recommended for security)
import os
hf_token = os.getenv("HF_TOKEN", None)

if hf_token:
    print("HF Token found")
else:
    print("Warning: No HF Token provided. You may encounter rate limits or access issues.")

In [ ]:
# Import necessary libraries
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import bitsandbytes as bnb
from datasets import load_dataset, Dataset, load_from_disk
import mlflow
import mlflow.pytorch
from datetime import datetime

# Check if GPU is available
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## Step 2: Load the Base Model and Tokenizer

We'll load a base model suitable for instruction tuning. For this workshop, we'll use TinyLlama or Phi-2 as examples.

In [ ]:
# Model configuration
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # You can also try "microsoft/phi-2"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token,  # Use HF token if provided
)
tokenizer.pad_token = tokenizer.eos_token  # Set pad token to eos token
tokenizer.padding_side = "right"  # Important for batching

# Load model with quantization for memory efficiency
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",  # Automatically map model to available devices
    token=hf_token,  # Use HF token if provided
)

print(f"Model loaded: {model_name}")
print(f"Model size: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M parameters")

## Step 3: Prepare Model for LoRA Training

We need to prepare the model for k-bit training (if using) and then apply LoRA configuration.

In [ ]:
# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=16,  # Rank
    lora_alpha=32,  # Alpha scaling
    target_modules=["q_proj", "v_proj"],  # Target modules for LoRA
    lora_dropout=0.05,  # Dropout probability
    bias="none",  # Bias type
    task_type="CAUSAL_LM"  # Task type
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

## Step 4: Load and Prepare Dataset

Now we'll load our prepared dataset from Exercise 2 and prepare it for training.

In [ ]:
# Load processed dataset from Exercise 2
from pathlib import Path
from datasets import load_from_disk

data_dir = Path("./data")
train_dataset = load_from_disk(str(data_dir / "train"))
eval_dataset = load_from_disk(str(data_dir / "validation"))

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(eval_dataset)}")

# Show a sample
print("\nSample from training data:")
print(train_dataset[0]["text"][:200])


## Step 5: Set Up Training Arguments and Tokenize Dataset

We need to tokenize our dataset and set up the training configuration.

In [ ]:
# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask"])
tokenized_eval.set_format(type="torch", columns=["input_ids", "attention_mask"])

# Set labels (for causal LM, labels = input_ids)
tokenized_train = tokenized_train.map(lambda x: {"labels": x["input_ids"]})
tokenized_eval = tokenized_eval.map(lambda x: {"labels": x["input_ids"]})

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    warmup_steps=10,
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    report_to="mlflow",
)

print("Training arguments configured successfully!")

## Step 6: Initialize Trainer and Start Training

Now we'll create the Trainer and start the training process.

In [ ]:
# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=lambda data: {
        'input_ids': torch.stack([x['input_ids'] for x in data]),
        'attention_mask': torch.stack([x['attention_mask'] for x in data]),
        'labels': torch.stack([x['labels'] for x in data])
    }
)

# Start MLflow run
mlflow.set_experiment("llm-lora-tuning")

# Train the model
print("Starting LoRA tuning...")
trainer.train()

# Save the LoRA adapter
trainer.save_model("./lora_adapter")

print("LoRA tuning completed and adapter saved!")

## Step 7: Save and Load LoRA Adapter

Let's see how to save and load our LoRA adapter for later use.

In [ ]:
# Save LoRA adapter
model.save_pretrained("./lora_adapter_final")
tokenizer.save_pretrained("./lora_adapter_final")

print("LoRA adapter saved successfully!")

# To load the adapter later:
# from peft import PeftModel
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
# )

# base_model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map="auto",
# )
# base_model = AutoModelForCausalLM.from_pretrained(model_name, load_in_4bit=True, device_map="auto")
# model = PeftModel.from_pretrained(base_model, "./lora_adapter_final")
# model = model.merge_and_unload()  # Optional: merge LoRA weights with base model

## Step 8: Test the Fine-tuned Model

Let's generate some text with our fine-tuned model to see if it learned anything.

In [ ]:
# Test generation with the fine-tuned model
from transformers import GenerationConfig

# Set model to evaluation mode
model.eval()

# Test prompt (using same format as training data)
test_prompt = """### Human: What is the importance of experiment tracking in MLOps?

### Assistant:
"""

# Tokenize input
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

# Generate
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode and print
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Generated response:")
print(response)


## Summary

In this exercise, we learned how to:
1. Load a base model with quantization for memory efficiency
2. Configure and apply LoRA for parameter-efficient fine-tuning
3. Prepare a dataset for instruction tuning
4. Set up training with MLflow tracking
5. Train the model using LoRA adapters
6. Save and load the LoRA adapter for deployment
7. Generate text with the fine-tuned model

## Next Steps

In the next exercise, we will evaluate our fine-tuned model using various metrics and compare it with the base model.

---
*Note: This is a simplified example for educational purposes. In a real workshop, you would use a larger, properly formatted dataset and train for more epochs.*